# 01 — Data cleaning

Six CSV files, one movie per row in `movies.csv`, and the other files hanging off `movie_id`. I load them, check what is actually in there, drop exact duplicate rows, and write a SQLite database.

Run this notebook from the project folder so the `data/` path works.


In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## Load and inspect


In [2]:
files = {
    "movies": "data/movies.csv",
    "genres": "data/genres.csv",
    "movie_genres": "data/movie_genres.csv",
    "cast": "data/cast.csv",
    "crew": "data/crew.csv",
    "movie_keywords": "data/movie_keywords.csv",
}

raw = {}
for name, path in files.items():
    df = pd.read_csv(path)
    raw[name] = df
    print("=" * 60)
    print(name, df.shape)
    print(df.dtypes)
    print("nulls:")
    print(df.isna().sum())
    print("exact duplicate rows:", int(df.duplicated().sum()))
    display(df.head(3))


movies (2509, 12)
movie_id               int64
title                    str
original_language        str
release_date             str
runtime                int64
budget               float64
revenue              float64
popularity           float64
vote_average         float64
vote_count             int64
status                   str
overview                 str
dtype: object
nulls:
movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64
exact duplicate rows: 6


,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...


genres (19, 2)
genre_id      int64
genre_name      str
dtype: object
nulls:
genre_id      0
genre_name    0
dtype: int64
exact duplicate rows: 0


,genre_id,genre_name
0,28,Action
1,12,Adventure
2,16,Animation


movie_genres (6888, 2)
movie_id    int64
genre_id    int64
dtype: object
nulls:
movie_id    0
genre_id    0
dtype: int64
exact duplicate rows: 10


...

,cast_row_id,movie_id,person_id,actor_name,character_name,cast_order
0,1,157336,10297,Matthew McConaughey,Cooper,0
1,2,157336,1813,Anne Hathaway,Brand,1
2,3,157336,3895,Michael Caine,Professor Brand,2


crew (7297, 6)
crew_row_id    int64
movie_id       int64
person_id      int64
person_name      str
job              str
department       str
dtype: object
nulls:
crew_row_id    0
movie_id       0
person_id      0
person_name    0
job            0
department     0
dtype: int64
exact duplicate rows: 8


,crew_row_id,movie_id,person_id,person_name,job,department
0,1,157336,527,Jonathan Nolan,Writer,Writing
1,2,157336,525,Christopher Nolan,Writer,Writing
2,3,157336,525,Christopher Nolan,Director,Directing


movie_keywords (41701, 3)
movie_id        int64
keyword_id      int64
keyword_name      str
dtype: object
nulls:
movie_id        0
keyword_id      0
keyword_name    0
dtype: int64
exact duplicate rows: 20


,movie_id,keyword_id,keyword_name
0,157336,1612,spacecraft
1,157336,4776,race against time
2,157336,310,artificial intelligence (a.i.)


## What I noticed before changing anything

- `movies` has 2,509 rows and 6 of them are exact copies. The repeated ids are Gladiator, Batman Forever, Doom, The Vow, Zoolander 2 and Anna. Same movie twice, not two different films.
- `movie_id` is the key. Some *titles* repeat with different ids (remakes). I am not dropping those here.
- Budget and revenue use `0` when TMDB doesn't have a number. That is not a free movie. I leave the zeros in the table.
- `release_date` came in as text. Cast already has `cast_row_id`, crew already has `crew_row_id`.
- 11 cast rows have no character name. I left those blank.
- No orphan ids in the raw files (every cast/crew/genre/keyword movie id exists). I still check again after the duplicate drop, in case a later edit removes a movie.


## Clean


In [3]:
movies = raw["movies"].drop_duplicates().copy()
genres = raw["genres"].drop_duplicates().copy()
movie_genres = raw["movie_genres"].drop_duplicates().copy()
cast = raw["cast"].drop_duplicates().copy()
crew = raw["crew"].drop_duplicates().copy()
keywords = raw["movie_keywords"].drop_duplicates().copy()

movies["release_date"] = pd.to_datetime(movies["release_date"])
movies["budget"] = movies["budget"].round().astype("int64")
movies["revenue"] = movies["revenue"].round().astype("int64")

# drop links that point at a movie or genre we no longer have
movie_ids = set(movies["movie_id"])
genre_ids = set(genres["genre_id"])

def drop_orphans(df, label):
    before = len(df)
    df = df[df["movie_id"].isin(movie_ids)].copy()
    print(f"{label}: removed {before - len(df)} orphan rows")
    return df

cast = drop_orphans(cast, "cast")
crew = drop_orphans(crew, "crew")
keywords = drop_orphans(keywords, "keywords")

before = len(movie_genres)
movie_genres = movie_genres[
    movie_genres["movie_id"].isin(movie_ids) & movie_genres["genre_id"].isin(genre_ids)
].copy()
print(f"movie_genres: removed {before - len(movie_genres)} orphan rows")

print("movies left", len(movies), "| unique movie_id", movies["movie_id"].nunique())
print("budget == 0:", int((movies.budget == 0).sum()), "| revenue == 0:", int((movies.revenue == 0).sum()))


cast: removed 0 orphan rows
crew: removed 0 orphan rows
keywords: removed 0 orphan rows
movie_genres: removed 0 orphan rows
movies left 2503 | unique movie_id 2503
budget == 0: 90 | revenue == 0: 102


## Load into SQLite

I used SQLite so the database is just a file in `data/` and nobody has to install MySQL to rerun this. Tables follow the column list in the project brief. `movie_genres` keeps the csv order, which matters later when I pick a movie's first genre. SQLite `rowid` will match that insert order.


In [4]:
# store dates as YYYY-MM-DD so SQL string comparison works
movies_sql = movies.copy()
movies_sql["release_date"] = movies_sql["release_date"].dt.strftime("%Y-%m-%d")

conn = sqlite3.connect("data/tmdb.db")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(
    """
    DROP TABLE IF EXISTS movie_keywords;
    DROP TABLE IF EXISTS "cast";
    DROP TABLE IF EXISTS crew;
    DROP TABLE IF EXISTS movie_genres;
    DROP TABLE IF EXISTS movies;
    DROP TABLE IF EXISTS genres;

    CREATE TABLE genres (
        genre_id INTEGER PRIMARY KEY,
        genre_name TEXT
    );

    CREATE TABLE movies (
        movie_id INTEGER PRIMARY KEY,
        title TEXT,
        original_language TEXT,
        release_date TEXT,
        runtime INTEGER,
        budget INTEGER,
        revenue INTEGER,
        popularity REAL,
        vote_average REAL,
        vote_count INTEGER,
        status TEXT,
        overview TEXT
    );

    CREATE TABLE movie_genres (
        movie_id INTEGER,
        genre_id INTEGER,
        PRIMARY KEY (movie_id, genre_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
    );

    CREATE TABLE "cast" (
        cast_row_id INTEGER PRIMARY KEY,
        movie_id INTEGER,
        person_id INTEGER,
        actor_name TEXT,
        character_name TEXT,
        cast_order INTEGER,
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );

    CREATE TABLE crew (
        crew_row_id INTEGER PRIMARY KEY,
        movie_id INTEGER,
        person_id INTEGER,
        person_name TEXT,
        job TEXT,
        department TEXT,
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );

    CREATE TABLE movie_keywords (
        movie_id INTEGER,
        keyword_id INTEGER,
        keyword_name TEXT,
        PRIMARY KEY (movie_id, keyword_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );
    """
)

genres.to_sql("genres", conn, if_exists="append", index=False)
movies_sql.to_sql("movies", conn, if_exists="append", index=False)
movie_genres.to_sql("movie_genres", conn, if_exists="append", index=False)
cast.to_sql("cast", conn, if_exists="append", index=False)
crew.to_sql("crew", conn, if_exists="append", index=False)
keywords.to_sql("movie_keywords", conn, if_exists="append", index=False)
conn.commit()

print("rows in sqlite:")
for table in ["movies", "genres", "movie_genres", "cast", "crew", "movie_keywords"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {n}")


rows in sqlite:
  movies: 2503
  genres: 19
  movie_genres: 6878
  cast: 24902
  crew: 7289
  movie_keywords: 41681


## Test join

One join across movies, movie_genres and genres, just to see that the keys line up before the SQL notebook.


In [5]:
check = pd.read_sql(
    """
    SELECT m.title, m.release_date, g.genre_name
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON g.genre_id = mg.genre_id
    ORDER BY m.title
    LIMIT 12
    """,
    conn,
)
check


,title,release_date,genre_name
0,(500) Days of Summer,2009-07-17,Comedy
1,(500) Days of Summer,2009-07-17,Drama
2,(500) Days of Summer,2009-07-17,Romance
3,10 Cloverfield Lane,2016-03-10,Thriller
4,10 Cloverfield Lane,2016-03-10,Science Fiction
5,10 Cloverfield Lane,2016-03-10,Drama
6,10 Cloverfield Lane,2016-03-10,Horror
7,10 Things I Hate About You,1999-03-30,Comedy
8,10 Things I Hate About You,1999-03-30,Romance
9,10 Things I Hate About You,1999-03-30,Drama


The join returns titles with their genres, so the keys are fine. 2,503 movies went into `data/tmdb.db` after the 6 duplicate rows came out. Orphan check removed nothing. Zeros in budget and revenue are still there on purpose.
